# tp-24 穴馬予測モデル v2

## 正解ラベル
- **条件 A**: 5着以内 かつ レース内の上位5頭で最も単勝オッズが高い（= 人気薄好走）
- **条件 B**: 3着以内 かつ 人気順位 6位以下

## マイナス評価クラス（`failed_fav`）
- 1〜5人気 かつ 5着外 = 人気馬が飛んだ馬。このクラスへの高予測にカスタム損失でペナルティを与える。

## モデル構成
| # | モデル | 目的関数 |
|---|---|---|
| M1 | LightGBM | カスタム非対称損失（失敗人気馬ペナルティ付き）|
| M2 | LightGBM | カスタム + Focal 項 |
| M3 | XGBoost | 加重 BCE（sample_weight） |
| Ens | rank 平均 | M1+M2+M3 OOF の rank 平均 |

## 特徴量
- layer_a の歴史系特徴量（オッズ・人気・着順 を除く）
- **ensemble_v2 / v3 OOF** p_win / p_top2 / p_top3（2022-2024 のみ有効、2020-2021 は NaN）
- レース内相対特徴量（p_win_rank, p_win_vs_top, 頭数など）

## 評価
- LOYO 2022〜2024（2020-2021 は常にトレーニングに含む）
- test 2025 は lock ファイルで一度だけ

In [1]:
import os, sys
from pathlib import Path

def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")

REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

PosixPath('/home/jovyan/work/keiba-vpn/.env.stg')

In [2]:
import json, subprocess, time
from datetime import datetime
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import mlflow
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score, average_precision_score

from src.pipeline.mlflow.train_utils import require_tracking_server

WORK    = REPO_ROOT / "data/local/modeling/training_pipeline"
BET     = REPO_ROOT / "data/local/modeling/betting"
OOF_DIR = WORK / "oof/model_oof"
OUT     = WORK / "upset"
OUT.mkdir(parents=True, exist_ok=True)
LOCK       = OUT / "TEST2025_EVALUATED.json"
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
SMOKE   = os.environ.get("TP24_SMOKE") == "1"
THREADS = int(os.environ.get("TP24_THREADS", "8"))
mlflow.set_experiment("keiba-tp-upset")
require_tracking_server()

YEARS_TRAIN_EXTRA = ["2020", "2021"]
YEARS_VALID       = ["2022", "2023", "2024"]
YEAR_TEST         = "2025"
YEARS_ALL         = YEARS_TRAIN_EXTRA + YEARS_VALID + [YEAR_TEST]
YEARS_TRAIN_ALL   = YEARS_TRAIN_EXTRA + YEARS_VALID

# カスタム損失のハイパーパラメータ
W_POS      = 4.0   # 正例（穴馬）重み
W_NEG_FAV  = 3.0   # 失敗人気馬（マイナス評価）重み
W_NEG      = 1.0   # その他の負例重み
ALPHA_FAV  = 2.0   # 失敗人気馬への追加勾配ペナルティ係数
GAMMA_FOCAL = 0.5  # Focal 損失の γ（小さくして安定化: 1.5→0.5）

2026-10-27 22:59:57 [INFO] MLflow: MLflow 接続: http://127.0.0.1:23230


## 1. データ読み込み・ラベル生成

In [3]:
layer_a = pd.read_parquet(WORK / "layer_a.parquet")
wp      = pd.read_parquet(BET  / "odds_win_place.parquet")

layer_a["year"] = layer_a["date"].str[:4]
df = layer_a[layer_a["year"].isin(YEARS_ALL)].copy()

df = df.merge(
    wp[["race_id", "horse_id", "win_odds", "popularity"]],
    on=["race_id", "horse_id"], how="left"
)
df = df.dropna(subset=["finish_position", "win_odds", "popularity"]).copy()
df["finish_position"] = df["finish_position"].astype(int)
df["popularity"]      = df["popularity"].astype(int)

# 正解ラベル
top5 = df[df["finish_position"] <= 5]
max_top5 = top5.groupby("race_id")["win_odds"].max().rename("_max_top5_odds")
df = df.merge(max_top5, on="race_id", how="left")
df["cond_a"] = (df["finish_position"] <= 5) & (df["win_odds"] == df["_max_top5_odds"])
df["cond_b"] = (df["finish_position"] <= 3) & (df["popularity"] >= 6)
df["label"]  = (df["cond_a"] | df["cond_b"]).astype(int)

# マイナス評価クラス: 1〜5人気 かつ 5着外
df["failed_fav"] = ((df["popularity"] <= 5) & (df["finish_position"] > 5)).astype(int)

# サンプル重み
df["sw"] = np.where(
    df["label"] == 1,   W_POS,
    np.where(df["failed_fav"] == 1, W_NEG_FAV, W_NEG)
)

print(f"行数: {len(df):,}  レース数: {df['race_id'].nunique():,}")
print(f"正例率: {df['label'].mean():.4f}  失敗人気馬率: {df['failed_fav'].mean():.4f}")
print("\n年別:")
print(df.groupby("year").agg(
    rows=("label","count"), pos=("label","sum"),
    failed_fav=("failed_fav","sum"), races=("race_id","nunique")
))

行数: 283,714  レース数: 20,733
正例率: 0.1068  失敗人気馬率: 0.1320

年別:
       rows   pos  failed_fav  races
year                                
2020  47876  5172        6431   3456
2021  47476  5093        6319   3456
2022  46840  4932        6112   3456
2023  47273  5047        6212   3456
2024  46752  5000        6165   3454
2025  47497  5067        6203   3455


## 2. レース内相対特徴（field_size）

**注**: ensemble_v2/v3 OOF meta 特徴量は想定オッズ代替のため除外済み。

In [4]:
# OOF meta 特徴量（v2/v3 確率）は除外
# field_size は layer_a から取得済み
assert 'field_size' in df.columns, 'field_size が layer_a に存在しません'
print(f'field_size OK: {df["field_size"].value_counts().head(3).to_dict()}')
print('OOF meta 特徴量 除外済み（v2/v3 確率なし）')


field_size OK: {16: 105855, 15: 31954, 14: 29943}
OOF meta 特徴量 除外済み（v2/v3 確率なし）


## 2.5 条件別騎手・調教師統計 + 複合スコア（jt_race_features）

`jt_race_features.parquet` から以下を追加:
- **条件別成績** : 距離帯・グレード・馬場・競馬場別の勝率・複勝率（jk/tr_at_*）
- **複合スコア** : `win_rate × 2 + top3_rate` で 1 指標に集約（重複特徴量削減）
- **直近フォーム** : 直近 10 戦の複勝率（roll10）
- **脚質指標** : 平均コーナー通過順位（jk_prior_all_avg_pass_norm_first）

In [5]:
JT_BASE = REPO_ROOT / "data/local/features/race_horse_tbl"

jt_dfs = []
for yr in YEARS_ALL:
    p = JT_BASE / yr / "jt_race_features.parquet"
    if p.exists():
        jt_dfs.append(pd.read_parquet(p, columns=[
            "race_id", "horse_id",
            # 距離帯別
            "jk_at_dist_win_rate", "jk_at_dist_top3_rate", "jk_at_dist_starts",
            "tr_at_dist_win_rate", "tr_at_dist_top3_rate", "tr_at_dist_starts",
            # グレード別
            "jk_at_grade_win_rate", "jk_at_grade_top3_rate", "jk_at_grade_starts",
            "tr_at_grade_win_rate", "tr_at_grade_top3_rate", "tr_at_grade_starts",
            # 馬場種別
            "jk_at_surface_win_rate", "jk_at_surface_top3_rate",
            "tr_at_surface_win_rate", "tr_at_surface_top3_rate",
            # 競馬場別
            "jk_at_venue_win_rate", "jk_at_venue_top3_rate",
            "tr_at_venue_win_rate", "tr_at_venue_top3_rate",
            # 馬場状態別
            "jk_at_track_cond_win_rate", "jk_at_track_cond_top3_rate",
            # 直近フォーム (カウント → rate を自前計算)
            "jk_roll10_wins", "jk_roll10_top3", "jk_roll10_starts",
            "tr_roll10_wins", "tr_roll10_top3", "tr_roll10_starts",
            "jk_roll5_wins",  "jk_roll5_top3",  "jk_roll5_starts",
            # 脚質: 平均コーナー通過順位（正規化）
            "jk_prior_all_avg_pass_norm_first",
            "jk_prior_all_avg_pass_first",
        ]))

jt = pd.concat(jt_dfs, ignore_index=True)

# rate を安全に計算
def _safe_rate(top, starts, min_starts=3):
    rate = np.where(starts >= min_starts, top / starts.clip(1), np.nan)
    return rate.astype("float32")

jt["jk_roll10_win_rate"]  = _safe_rate(jt["jk_roll10_wins"],  jt["jk_roll10_starts"])
jt["jk_roll10_top3_rate"] = _safe_rate(jt["jk_roll10_top3"],  jt["jk_roll10_starts"])
jt["jk_roll5_top3_rate"]  = _safe_rate(jt["jk_roll5_top3"],   jt["jk_roll5_starts"])
jt["tr_roll10_win_rate"]  = _safe_rate(jt["tr_roll10_wins"],   jt["tr_roll10_starts"])
jt["tr_roll10_top3_rate"] = _safe_rate(jt["tr_roll10_top3"],   jt["tr_roll10_starts"])

# 複合スコア (win_rate × 2 + top3_rate)
for prefix in ("jk", "tr"):
    for cond in ("all", "dist", "grade", "surface", "venue"):
        if cond == "all":
            # layer_a のグローバル率から生成 (jt_race_features には jk_prior_all_win_rate あるが layer_a 既存)
            continue
        w = jt.get(f"{prefix}_at_{cond}_win_rate")
        t3 = jt.get(f"{prefix}_at_{cond}_top3_rate")
        if w is not None and t3 is not None:
            jt[f"{prefix}_score_{cond}"] = (w * 2.0 + t3).astype("float32")

# 騎手複合スコア（グローバル）は layer_a の既存列から df に追加後に計算
jt_cols_keep = ["race_id", "horse_id",
    "jk_at_dist_win_rate", "jk_at_dist_top3_rate",
    "jk_at_grade_win_rate", "jk_at_grade_top3_rate",
    "jk_at_surface_win_rate", "jk_at_surface_top3_rate",
    "jk_at_venue_win_rate", "jk_at_venue_top3_rate",
    "jk_at_track_cond_win_rate", "jk_at_track_cond_top3_rate",
    "tr_at_dist_win_rate", "tr_at_dist_top3_rate",
    "tr_at_grade_win_rate", "tr_at_grade_top3_rate",
    "tr_at_surface_win_rate", "tr_at_surface_top3_rate",
    "tr_at_venue_win_rate", "tr_at_venue_top3_rate",
    "jk_roll10_win_rate", "jk_roll10_top3_rate",
    "jk_roll5_top3_rate",
    "tr_roll10_win_rate", "tr_roll10_top3_rate",
    "jk_prior_all_avg_pass_norm_first",
    "jk_score_dist", "jk_score_grade", "jk_score_surface", "jk_score_venue",
    "tr_score_dist", "tr_score_grade", "tr_score_surface", "tr_score_venue",
]
jt_cols_keep = [c for c in jt_cols_keep if c in jt.columns]

df = df.merge(jt[jt_cols_keep], on=["race_id", "horse_id"], how="left")

# グローバル複合スコア（layer_a 既存の jockey_prior_win_rate から）
df["jk_score_all"] = (df["jockey_prior_win_rate"] * 2.0 + df["jockey_prior_top3_rate"]).astype("float32")
df["tr_score_all"] = (df["trainer_prior_win_rate"] * 2.0 + df["trainer_prior_top3_rate"]).astype("float32")

# 条件別スコアの「グローバルからの乖離」（その条件での得意度）
df["jk_dist_vs_all"]    = df["jk_score_dist"]    - df["jk_score_all"]
df["tr_dist_vs_all"]    = df["tr_score_dist"]    - df["tr_score_all"]
df["jk_grade_vs_all"]   = df["jk_score_grade"]   - df["jk_score_all"]
df["jk_surface_vs_all"] = df["jk_score_surface"] - df["jk_score_all"]
df["jk_venue_vs_all"]   = df["jk_score_venue"]   - df["jk_score_all"]

# 直近フォーム比: roll10 / global（直近の調子指標）
df["jk_form_ratio"] = df["jk_roll10_win_rate"] / df["jockey_prior_win_rate"].replace(0, np.nan)
df["tr_form_ratio"] = df["tr_roll10_win_rate"] / df["trainer_prior_win_rate"].replace(0, np.nan)

new_jt_cols = [c for c in df.columns if c in jt_cols_keep and c not in ("race_id","horse_id")]
new_jt_cols += ["jk_score_all", "tr_score_all",
                "jk_dist_vs_all", "tr_dist_vs_all",
                "jk_grade_vs_all", "jk_surface_vs_all", "jk_venue_vs_all",
                "jk_form_ratio", "tr_form_ratio"]

print(f"jt_race_features 追加列: {len(new_jt_cols)}")
miss_jt = df[new_jt_cols].isnull().mean()
print("欠損率 >10% の列:")
print(miss_jt[miss_jt > 0.10].sort_values(ascending=False).head(10).round(3))

jt_race_features 追加列: 41
欠損率 >10% の列:
Series([], dtype: float64)


## 2.6 走法スコア（tp-25）+ レース質特徴量（tp-26）+ 血統特徴量

In [6]:
LS_PATH  = WORK / "lapscores/lapscores.parquet"
RQ_PATH  = WORK / "race_quality/horse_rq_features.parquet"
PED_PATH = REPO_ROOT / "data/page_reference/pedigree_race_index/horse_sire_damsire.parquet"

# ── tp-25: 走法スコア ──────────────────────────────────────────
ls_cols = ["race_id","horse_id",
           "ts_score_avg3","ts_score_avg5","ts_score_last1",
           "gc_score_avg3","gc_score_avg5","gc_score_last1"]
if LS_PATH.exists():
    ls = pd.read_parquet(LS_PATH, columns=ls_cols)
    df = df.merge(ls, on=["race_id","horse_id"], how="left")
    print(f"tp-25 走法スコア: {len(ls_cols)-2} 列追加  欠損率: {df['ts_score_avg3'].isna().mean():.1%}")
else:
    print("WARNING: lapscores.parquet not found")

# ── tp-26: レース質 走法プロファイル特徴量 ──────────────────────
rq_cols = ["race_id","horse_id",
           "split_ratio_avg3","split_ratio_avg5","split_ratio_last1",
           "front_ratio_avg3","front_ratio_avg5","front_ratio_last1",
           "shift_ratio_avg3",
           "rq_compat","rq_compat_dev"]
if RQ_PATH.exists():
    rq_feat = pd.read_parquet(RQ_PATH, columns=rq_cols)
    df = df.merge(rq_feat, on=["race_id","horse_id"], how="left")
    print(f"tp-26 レース質特徴量: {len(rq_cols)-2} 列追加  欠損率: {df['split_ratio_avg3'].isna().mean():.1%}")
else:
    print("WARNING: horse_rq_features.parquet not found")

# ── 血統: 時系列安全なターゲットエンコーディング ────────────────
# 生カテゴリ(766種牡馬)の代わりに、date < レース日 の過去産駒勝率・複勝率を使用
if PED_PATH.exists():
    ped = pd.read_parquet(PED_PATH, columns=["horse_id","sire","dam_sire"])
    for col in ["sire","dam_sire"]:
        ped[col] = ped[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan})
    ped = ped.drop_duplicates("horse_id")

    dfr = df[["race_id","horse_id","date","finish_position"]].copy()
    dfr = dfr.merge(ped, on="horse_id", how="left")
    dfr["date_dt"] = pd.to_datetime(dfr["date"])
    dfr["win"]  = (dfr["finish_position"] == 1).astype(float)
    dfr["top3"] = (dfr["finish_position"] <= 3).astype(float)

    def _rolling_sire_feat(dfr, sire_col, min_starts=10):
        """date < レース日 の産駒勝率・複勝率を時系列安全に計算。同日は除外。"""
        valid = dfr[dfr[sire_col].notna()].copy()
        daily = valid.groupby([sire_col, "date_dt"]).agg(
            d_wins=("win",  "sum"),
            d_top3=("top3", "sum"),
            d_cnt =("win",  "count"),
        ).reset_index().sort_values([sire_col, "date_dt"])
        daily["cum_wins"] = daily.groupby(sire_col)["d_wins"].cumsum()
        daily["cum_top3"] = daily.groupby(sire_col)["d_top3"].cumsum()
        daily["cum_cnt"]  = daily.groupby(sire_col)["d_cnt"].cumsum()
        for c in ["cum_wins","cum_top3","cum_cnt"]:
            daily[f"prev_{c}"] = daily.groupby(sire_col)[c].shift(1).fillna(0)
        daily[f"{sire_col}_win_rate"]  = np.where(
            daily["prev_cum_cnt"] >= min_starts,
            daily["prev_cum_wins"] / daily["prev_cum_cnt"].clip(1), np.nan
        ).astype("float32")
        daily[f"{sire_col}_top3_rate"] = np.where(
            daily["prev_cum_cnt"] >= min_starts,
            daily["prev_cum_top3"] / daily["prev_cum_cnt"].clip(1), np.nan
        ).astype("float32")
        feat_cols = [sire_col, "date_dt",
                     f"{sire_col}_win_rate", f"{sire_col}_top3_rate"]
        return valid.merge(daily[feat_cols], on=[sire_col, "date_dt"], how="left")\
                    [["race_id","horse_id",
                      f"{sire_col}_win_rate", f"{sire_col}_top3_rate"]]

    sire_feat     = _rolling_sire_feat(dfr, "sire")
    dam_sire_feat = _rolling_sire_feat(dfr, "dam_sire")

    df = df.drop(columns=["sire","dam_sire"], errors="ignore")
    df = df.merge(sire_feat,     on=["race_id","horse_id"], how="left")
    df = df.merge(dam_sire_feat, on=["race_id","horse_id"], how="left")

    print(f"sire_win_rate      有効率: {df['sire_win_rate'].notna().mean():.1%}"
          f"  平均: {df['sire_win_rate'].mean():.4f}")
    print(f"dam_sire_top3_rate 有効率: {df['dam_sire_top3_rate'].notna().mean():.1%}"
          f"  平均: {df['dam_sire_top3_rate'].mean():.4f}")
else:
    print("WARNING: horse_sire_damsire.parquet not found")

print(f"\n現在の df 列数: {len(df.columns)}")

tp-25 走法スコア: 6 列追加  欠損率: 18.4%
tp-26 レース質特徴量: 9 列追加  欠損率: 18.5%
sire_win_rate      有効率: 97.7%  平均: 0.0776
dam_sire_top3_rate 有効率: 96.3%  平均: 0.2236

現在の df 列数: 138


## 3. 特徴量の準備

In [7]:
EXCLUDE = {
    "race_id","horse_id","horse_name","jockey_name","jockey_id",
    "trainer_name","trainer_id","race_name",
    "date","year","start_time",
    "win_odds","popularity",
    "finish_position",
    "past_races","training",
    "cond_a","cond_b","label","failed_fav","sw","_max_top5_odds",
    # rq_compat は偏差値版 rq_compat_dev に集約済みなので除外
    "rq_compat",
}
FEATS = [c for c in df.columns if c not in EXCLUDE]

# object & category dtype 両方を CAT_COLS に含める
CAT_COLS = [c for c in FEATS if df[c].dtype == object or hasattr(df[c].dtype, "categories")]
for c in CAT_COLS:
    if df[c].dtype == object:
        df[c] = df[c].astype("category")

print(f"特徴量数: {len(FEATS)}  カテゴリ列: {len(CAT_COLS)} 列 → {CAT_COLS[:8]}...")
miss = df[FEATS].isnull().mean()
print("欠損率 >5%:\n", miss[miss > 0.05].sort_values(ascending=False).head(15).round(4))


def to_xgb(X: pd.DataFrame) -> np.ndarray:
    """XGBoost 用: category 列を整数コード（NaN=-1）に変換して float32 numpy array を返す。"""
    X2 = X.copy()
    for c in X2.select_dtypes(["category"]).columns:
        X2[c] = X2[c].cat.codes.astype("float32")
        X2.loc[X2[c] == -1, c] = np.nan
    return X2.to_numpy(dtype=np.float32)

特徴量数: 115  カテゴリ列: 14 列 → ['venue', 'surface', 'direction', 'grade', 'race_class', 'weather', 'track_condition', 'venue_code']...
欠損率 >5%:
 sire_x_track_bias_diff           0.5770
track_bias_winner_3f_diff        0.5770
dam_sire_x_track_bias_diff       0.5770
track_bias_winner_3f_prev_day    0.4962
speed_recent_3                   0.3010
speed_recent_2                   0.2222
split_ratio_last1                0.1850
split_ratio_avg5                 0.1850
split_ratio_avg3                 0.1850
front_ratio_last1                0.1850
front_ratio_avg5                 0.1850
shift_ratio_avg3                 0.1850
front_ratio_avg3                 0.1850
ts_score_avg3                    0.1845
ts_score_avg5                    0.1845
dtype: float64


## 4. カスタム損失関数の定義

In [8]:
def _make_upset_obj(failed_fav_arr: np.ndarray, w_pos: float, w_neg_fav: float,
                    w_neg: float, alpha_fav: float, gamma: float = 0.0):
    """LightGBM 4.x 用カスタム目的関数を返すクロージャ。
    シグネチャ: (y_pred, dataset) -> (grad, hess)

    非対称 weighted BCE:
      - 正例（upset）             : w_pos × BCE
      - 失敗人気馬（failed_fav=1）: w_neg_fav × BCE + alpha_fav × p  ← 追加ペナルティ
      - その他負例                : w_neg × BCE
    gamma > 0 のとき Focal 重み (1-p)^gamma を乗算。
    """
    def objective(y_pred, dataset):
        y_true = dataset.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))
        p = np.clip(p, 1e-7, 1 - 1e-7)

        g = p - y_true
        h = p * (1.0 - p)

        if gamma > 0:
            pt = np.where(y_true == 1, p, 1.0 - p)
            focal_w = (1.0 - pt) ** gamma
            g *= focal_w
            h *= focal_w

        w = np.where(y_true == 1, w_pos,
            np.where(failed_fav_arr, w_neg_fav, w_neg)).astype(float)
        g *= w
        h *= w

        fav_neg = (y_true == 0) & failed_fav_arr.astype(bool)
        g[fav_neg] += alpha_fav * p[fav_neg]
        h[fav_neg] += alpha_fav * p[fav_neg] * (1.0 - p[fav_neg])

        return g, h
    return objective


def lgb_auc_eval(y_pred, dataset):
    """LightGBM 4.x 用カスタム評価指標 (AUC)。"""
    y_true = dataset.get_label()
    auc = roc_auc_score(y_true, y_pred)
    return "auc", auc, True


print("カスタム損失関数を定義しました（LightGBM 4.x シグネチャ）")

カスタム損失関数を定義しました（LightGBM 4.x シグネチャ）


## 5. モデル学習（LOYO 2022〜2024）

In [9]:
LGB_BASE = {
    "learning_rate": 0.03,
    "num_leaves": 127,
    "min_data_in_leaf": 40,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "lambda_l1": 0.1,
    "max_depth": -1,
    "verbosity": -1,
    "seed": 42,
    "deterministic": True,
    "num_threads": THREADS,
}
XGB_PARAMS = {
    "max_depth": 7,
    "learning_rate": 0.03,
    "n_estimators": 200 if SMOKE else 600,
    "subsample": 0.8,
    "colsample_bytree": 0.75,
    "reg_lambda": 2.0,
    "min_child_weight": 30,
    "tree_method": "hist",
    "device": "cpu",
    "seed": 42,
    "nthread": THREADS,
    "early_stopping_rounds": 50,
}
N_ROUNDS_LGB = 300 if SMOKE else 800

oof_m1 = np.zeros(len(df))
oof_m2 = np.zeros(len(df))
oof_m3 = np.zeros(len(df))
loyo_rows = []
all_curves = []

for val_year in YEARS_VALID:
    t0 = time.time()
    tr_years = [y for y in YEARS_TRAIN_ALL if y != val_year]
    tr_mask  = df["year"].isin(tr_years)
    va_mask  = df["year"] == val_year

    X_tr  = df.loc[tr_mask, FEATS]
    y_tr  = df.loc[tr_mask, "label"].to_numpy()
    ff_tr = df.loc[tr_mask, "failed_fav"].to_numpy()
    sw_tr = df.loc[tr_mask, "sw"].to_numpy()
    X_va  = df.loc[va_mask, FEATS]
    y_va  = df.loc[va_mask, "label"].to_numpy()

    # --- M1: LGB カスタム損失 ---
    obj1 = _make_upset_obj(ff_tr, W_POS, W_NEG_FAV, W_NEG, ALPHA_FAV, gamma=0.0)
    ev1  = {}
    ds_tr1 = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT_COLS, free_raw_data=False)
    ds_va1 = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT_COLS, reference=ds_tr1, free_raw_data=False)
    m1 = lgb.train(
        {**LGB_BASE, "objective": obj1},
        ds_tr1, num_boost_round=N_ROUNDS_LGB,
        valid_sets=[ds_va1], valid_names=["val"],
        feval=lgb_auc_eval,
        callbacks=[lgb.early_stopping(60, verbose=False, first_metric_only=True),
                   lgb.log_evaluation(period=150),
                   lgb.record_evaluation(ev1)],
    )
    oof_m1[va_mask.values] = m1.predict(X_va)
    for i, v in enumerate(ev1.get("val", {}).get("auc", [])):
        all_curves.append({"val_year": val_year, "model": "M1_lgb_custom", "round": i+1, "auc": v})

    # --- M2: LGB カスタム + Focal ---
    obj2 = _make_upset_obj(ff_tr, W_POS, W_NEG_FAV, W_NEG, ALPHA_FAV, gamma=GAMMA_FOCAL)
    ev2  = {}
    ds_tr2 = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT_COLS, free_raw_data=False)
    ds_va2 = lgb.Dataset(X_va, label=y_va, categorical_feature=CAT_COLS, reference=ds_tr2, free_raw_data=False)
    m2 = lgb.train(
        {**LGB_BASE, "objective": obj2},
        ds_tr2, num_boost_round=N_ROUNDS_LGB,
        valid_sets=[ds_va2], valid_names=["val"],
        feval=lgb_auc_eval,
        callbacks=[lgb.early_stopping(60, verbose=False, first_metric_only=True),
                   lgb.log_evaluation(period=150),
                   lgb.record_evaluation(ev2)],
    )
    oof_m2[va_mask.values] = m2.predict(X_va)
    for i, v in enumerate(ev2.get("val", {}).get("auc", [])):
        all_curves.append({"val_year": val_year, "model": "M2_lgb_focal", "round": i+1, "auc": v})

    # --- M3: XGBoost + sample_weight (category → 整数コード) ---
    Xtr_xgb = to_xgb(X_tr)
    Xva_xgb = to_xgb(X_va)
    xgb_tr   = xgb.DMatrix(Xtr_xgb, label=y_tr, weight=sw_tr, feature_names=FEATS)
    xgb_eval = xgb.DMatrix(Xva_xgb, label=y_va, feature_names=FEATS)
    ev3 = {}
    m3_xgb = xgb.train(
        {k: v for k, v in XGB_PARAMS.items() if k not in ("n_estimators","early_stopping_rounds")},
        xgb_tr,
        num_boost_round=XGB_PARAMS["n_estimators"],
        evals=[(xgb_eval, "val")],
        early_stopping_rounds=XGB_PARAMS["early_stopping_rounds"],
        evals_result=ev3,
        verbose_eval=False,
        custom_metric=lambda p, d: ("auc", float(roc_auc_score(d.get_label(), p))),
        maximize=True,
    )
    oof_m3[va_mask.values] = m3_xgb.predict(xgb.DMatrix(Xva_xgb, feature_names=FEATS))
    for i, v in enumerate(ev3.get("val", {}).get("auc", [])):
        all_curves.append({"val_year": val_year, "model": "M3_xgb", "round": i+1, "auc": v})

    # メトリクス
    from scipy.stats import rankdata as _rd
    auc1 = roc_auc_score(y_va, oof_m1[va_mask.values])
    auc2 = roc_auc_score(y_va, oof_m2[va_mask.values])
    auc3 = roc_auc_score(y_va, oof_m3[va_mask.values])
    ens  = (_rd(oof_m1[va_mask.values]) + _rd(oof_m2[va_mask.values]) + _rd(oof_m3[va_mask.values])) / 3
    auc_ens = roc_auc_score(y_va, ens)
    loyo_rows.append({
        "val_year": val_year,
        "auc_m1": auc1, "auc_m2": auc2, "auc_m3": auc3, "auc_ens": auc_ens,
        "best_iter_m1": m1.best_iteration,
        "best_iter_m2": m2.best_iteration,
        "best_iter_m3": m3_xgb.best_iteration,
        "n": int(va_mask.sum()), "sec": time.time() - t0,
    })
    print(f"{val_year}: M1={auc1:.4f}  M2={auc2:.4f}  M3={auc3:.4f}  Ens={auc_ens:.4f}  ({time.time()-t0:.0f}s)")

LOYO   = pd.DataFrame(loyo_rows)
CURVES = pd.DataFrame(all_curves)
CURVES.to_csv(OUT / "learning_curves.csv", index=False)
print("\n平均:")
print(LOYO[["auc_m1","auc_m2","auc_m3","auc_ens"]].mean().round(4))
print(f"\n学習曲線を保存: {OUT / 'learning_curves.csv'}")

[150]	val's auc: 0.642979
[300]	val's auc: 0.647584
[450]	val's auc: 0.649361
[600]	val's auc: 0.650107
[150]	val's auc: 0.638154
[300]	val's auc: 0.64209
[450]	val's auc: 0.642758
2022: M1=0.6502  M2=0.6430  M3=0.6554  Ens=0.6542  (106s)
[150]	val's auc: 0.634682
[300]	val's auc: 0.640463
[450]	val's auc: 0.642505
[150]	val's auc: 0.632391
[300]	val's auc: 0.63524
2023: M1=0.6426  M2=0.6366  M3=0.6484  Ens=0.6470  (92s)
[150]	val's auc: 0.639732
[300]	val's auc: 0.642132
[150]	val's auc: 0.635381
2024: M1=0.6423  M2=0.6365  M3=0.6483  Ens=0.6463  (62s)

平均:
auc_m1     0.6450
auc_m2     0.6387
auc_m3     0.6507
auc_ens    0.6491
dtype: float64

学習曲線を保存: /home/jovyan/work/keiba-vpn/data/local/modeling/training_pipeline/upset/learning_curves.csv


## 6. OOF アンサンブルスコアの評価

In [10]:
# 学習曲線を MLflow に記録（fold ごと × モデルごと）
with mlflow.start_run(run_name="tp-24-loyo-curves",
                      tags={"notebook": "tp-24", "git_commit": GIT_COMMIT}) as _run:
    for _, row in LOYO.iterrows():
        y = row["val_year"]
        mlflow.log_metrics({
            f"auc_m1_final_{y}": row["auc_m1"],
            f"auc_m2_final_{y}": row["auc_m2"],
            f"auc_m3_final_{y}": row["auc_m3"],
            f"auc_ens_final_{y}": row["auc_ens"],
        })
    # step ごとの AUC（モデル別・fold 別）
    for model_name, group in CURVES.groupby("model"):
        for val_year, sub in group.groupby("val_year"):
            for _, r in sub.iterrows():
                mlflow.log_metric(
                    f"auc_{model_name}_{val_year}", r["auc"], step=int(r["round"])
                )
    mlflow.log_artifact(str(OUT / "learning_curves.csv"), "upset")
    CURVE_RUN_ID = _run.info.run_id

print(f"学習曲線を MLflow に記録しました (run_id={CURVE_RUN_ID})")

🏃 View run tp-24-loyo-curves at: http://127.0.0.1:23230/#/experiments/26/runs/f1937b49158b4508b5d3fa3ea81235a5
🧪 View experiment at: http://127.0.0.1:23230/#/experiments/26
学習曲線を MLflow に記録しました (run_id=f1937b49158b4508b5d3fa3ea81235a5)


In [11]:
valid_mask = df["year"].isin(YEARS_VALID)
dv = df[valid_mask].copy().reset_index(drop=True)
idx_v = np.where(valid_mask.values)[0]

# rank 平均アンサンブルスコア
n_v = len(dv)
dv["score"] = (
    rankdata(oof_m1[idx_v]) +
    rankdata(oof_m2[idx_v]) +
    rankdata(oof_m3[idx_v])
) / 3.0

# レースごとスコア最大の馬を選ぶ
pick_idx = dv.groupby("race_id")["score"].idxmax()
picks = dv.loc[pick_idx]
print("=== レースごとにスコア最大の馬 ===")
print(f"的中率（正例率）: {picks['label'].mean():.4f}  ({picks['label'].sum()}/{len(picks)})")
print(f"  cond_A: {picks['cond_a'].mean():.4f}  cond_B: {picks['cond_b'].mean():.4f}")
print(f"win_odds 平均: {picks['win_odds'].mean():.1f}  popularity 平均: {picks['popularity'].mean():.1f}")

# 閾値別
print("\n--- スコアパーセンタイル別の適合率・再現率 ---")
rows = []
for pct in [80, 85, 90, 95]:
    th = np.percentile(dv["score"], pct)
    sel = dv[dv["score"] >= th]
    rows.append({
        "閾値": f">={pct}%ile", "選択数": len(sel),
        "適合率": round(sel["label"].mean(), 4),
        "再現率": round(sel["label"].sum() / dv["label"].sum(), 4),
        "win_odds平均": round(sel["win_odds"].mean(), 1),
        "popularity平均": round(sel["popularity"].mean(), 1),
    })
display(pd.DataFrame(rows))

# 失敗人気馬への予測が下がったか確認
print("\n--- 失敗人気馬 vs その他の平均スコア（正規化） ---")
dv["score_norm"] = dv["score"] / dv["score"].max()
print(dv.groupby("failed_fav")["score_norm"].agg(["mean","median"]).round(4))

=== レースごとにスコア最大の馬 ===
的中率（正例率）: 0.1775  (1840/10366)
  cond_A: 0.1451  cond_B: 0.0811
win_odds 平均: 97.8  popularity 平均: 9.9

--- スコアパーセンタイル別の適合率・再現率 ---


,閾値,選択数,適合率,再現率,win_odds平均,popularity平均
0,>=80%ile,28173,0.1696,0.3190,89.5,9.5
1,>=85%ile,21130,0.1765,0.2490,89.9,9.5
2,>=90%ile,14087,0.1850,0.1740,90.1,9.4
3,>=95%ile,7044,0.2032,0.0955,88.8,9.0



--- 失敗人気馬 vs その他の平均スコア（正規化） ---
              mean  median
failed_fav                
0           0.5236  0.5429
1           0.3440  0.2879


## 7. 最終モデルの学習（2020-2024 全データ）

In [12]:
tr_all_mask = df["year"].isin(YEARS_TRAIN_ALL)
X_all  = df.loc[tr_all_mask, FEATS]
y_all  = df.loc[tr_all_mask, "label"].to_numpy()
ff_all = df.loc[tr_all_mask, "failed_fav"].to_numpy()
sw_all = df.loc[tr_all_mask, "sw"].to_numpy()

best_m1 = int(LOYO["best_iter_m1"].mean())
best_m2 = int(LOYO["best_iter_m2"].mean())
best_m3 = int(LOYO["best_iter_m3"].mean())

obj1_all = _make_upset_obj(ff_all, W_POS, W_NEG_FAV, W_NEG, ALPHA_FAV, gamma=0.0)
obj2_all = _make_upset_obj(ff_all, W_POS, W_NEG_FAV, W_NEG, ALPHA_FAV, gamma=GAMMA_FOCAL)

ds_all1 = lgb.Dataset(X_all, label=y_all, categorical_feature=CAT_COLS, free_raw_data=False)
final_m1 = lgb.train({**LGB_BASE, "objective": obj1_all}, ds_all1, num_boost_round=best_m1)

ds_all2 = lgb.Dataset(X_all, label=y_all, categorical_feature=CAT_COLS, free_raw_data=False)
final_m2 = lgb.train({**LGB_BASE, "objective": obj2_all}, ds_all2, num_boost_round=best_m2)

Xall_xgb = to_xgb(X_all)
xgb_all  = xgb.DMatrix(Xall_xgb, label=y_all, weight=sw_all, feature_names=FEATS)
final_m3 = xgb.train(
    {k: v for k, v in XGB_PARAMS.items() if k not in ("n_estimators","early_stopping_rounds")},
    xgb_all, num_boost_round=best_m3, verbose_eval=False,
)

print(f"最終モデル学習完了: M1={best_m1}  M2={best_m2}  M3={best_m3} rounds")

imp = pd.Series(final_m1.feature_importance(importance_type="gain"), index=FEATS)
imp = imp.sort_values(ascending=False)
print("\nM1 特徴量重要度（gain）上位 25:")
display(imp.head(25).round(0).to_frame("gain"))

最終モデル学習完了: M1=433  M2=321  M3=402 rounds

M1 特徴量重要度（gain）上位 25:


,gain
track_bias_stage_key,302511.0
horse_prior_avg_finish,135740.0
horse_prior_top3_rate,90828.0
sex_age,75507.0
race_class,70519.0
speed_recent_1,48833.0
jockey_prior_win_rate,37413.0
front_ratio_avg3,36118.0
front_ratio_last1,35203.0
sire_win_rate,33215.0


## 8. 仮想回収率（valid 2022-2024）

In [13]:
pay = pd.read_parquet(BET / "payoff.parquet")
tansho  = pay[pay["bet_type"] == "tansho"][["race_id","combo","payout"]].copy()
fukusho = pay[pay["bet_type"] == "fukusho"][["race_id","combo","payout"]].copy()
tansho["combo"]  = tansho["combo"].astype(str)
fukusho["combo"] = fukusho["combo"].astype(str)
fukusho = fukusho.rename(columns={"payout": "place_payout"})

dv2 = dv.copy()
dv2["horse_number_s"] = dv2["horse_number"].astype(str)
dv2 = dv2.merge(tansho,  left_on=["race_id","horse_number_s"], right_on=["race_id","combo"], how="left")
dv2 = dv2.merge(fukusho, left_on=["race_id","horse_number_s"], right_on=["race_id","combo"], how="left")
dv2["payout"]       = dv2["payout"].fillna(0)
dv2["place_payout"] = dv2["place_payout"].fillna(0)

print("スコア上位 k 頭/レース の単勝・複勝回収率（valid 2022-2024）")
rows = []
for topk in [1, 2, 3]:
    sel = dv2.groupby("race_id", group_keys=False).apply(
        lambda g: g.nlargest(topk, "score")
    )
    rows.append({
        "上位k頭": topk,
        "単勝回収率": round(sel["payout"].mean() / 100, 4),
        "複勝回収率": round(sel["place_payout"].mean() / 100, 4),
        "買い目数": len(sel),
    })
T_roi = pd.DataFrame(rows)
display(T_roi)

rnd = dv2.groupby("race_id", group_keys=False).apply(lambda g: g.sample(1, random_state=0))
print(f"ランダム: 単勝={rnd['payout'].mean()/100:.4f}  複勝={rnd['place_payout'].mean()/100:.4f}")

スコア上位 k 頭/レース の単勝・複勝回収率（valid 2022-2024）


,上位k頭,単勝回収率,複勝回収率,買い目数
0,1,0.6594,0.8093,10366
1,2,0.6813,0.7870,20732
2,3,0.7026,0.7870,31098


ランダム: 単勝=0.7937  複勝=0.7323


## 9. test 2025（一度だけ）

In [14]:
if LOCK.exists() and os.environ.get("TP24_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）")

te_mask = df["year"] == YEAR_TEST
X_te = df.loc[te_mask, FEATS]
y_te = df.loc[te_mask, "label"].to_numpy()

p1_te = final_m1.predict(X_te)
p2_te = final_m2.predict(X_te)
Xte_xgb = to_xgb(X_te)
p3_te = final_m3.predict(xgb.DMatrix(Xte_xgb, feature_names=FEATS))
ens_te = (rankdata(p1_te) + rankdata(p2_te) + rankdata(p3_te)) / 3.0

auc_te = roc_auc_score(y_te, ens_te)
ap_te  = average_precision_score(y_te, ens_te)
print(f"test 2025: AUC={auc_te:.4f}  AP={ap_te:.4f}  n={len(y_te):,}  pos_rate={y_te.mean():.4f}")

dte = df[te_mask].copy().reset_index(drop=True)
dte["score"] = ens_te
dte["horse_number_s"] = dte["horse_number"].astype(str)
dte = dte.merge(tansho,  left_on=["race_id","horse_number_s"], right_on=["race_id","combo"], how="left")
dte = dte.merge(fukusho, left_on=["race_id","horse_number_s"], right_on=["race_id","combo"], how="left")
dte["payout"]       = dte["payout"].fillna(0)
dte["place_payout"] = dte["place_payout"].fillna(0)

rows_te = []
for topk in [1, 2, 3]:
    sel = dte.groupby("race_id", group_keys=False).apply(
        lambda g: g.nlargest(topk, "score")
    )
    rows_te.append({
        "上位k頭": topk,
        "単勝回収率": round(sel["payout"].mean() / 100, 4),
        "複勝回収率": round(sel["place_payout"].mean() / 100, 4),
    })
T_te = pd.DataFrame(rows_te)
display(T_te)

res = {
    "evaluated_at": datetime.now().isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT,
    "version": "tp-24-v5-no-oof",
    "new_features": ["ts_score_avg3/5","gc_score_avg3/5","split_ratio_avg3/5",
                     "front_ratio_avg3/5","rq_compat_dev",
                     "sire_win_rate","sire_top3_rate",
                     "dam_sire_win_rate","dam_sire_top3_rate"],
    "hyperparams": {"W_POS": W_POS, "W_NEG_FAV": W_NEG_FAV, "W_NEG": W_NEG,
                    "ALPHA_FAV": ALPHA_FAV, "GAMMA_FOCAL": GAMMA_FOCAL},
    "loyo": LOYO.to_dict("records"),
    "test2025": {"auc": auc_te, "ap": ap_te, "roi": T_te.to_dict("records")},
}
LOCK.write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str))
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str))

with mlflow.start_run(run_name="tp-24-upset-v5-no-oof",
                      tags={"notebook": "tp-24", "git_commit": GIT_COMMIT,
                            "feature_set": "lapscores+rq+sire_rolling+no_oof"}):
    mlflow.log_metrics({
        "loyo_auc_ens": float(LOYO["auc_ens"].mean()),
        "loyo_auc_m1":  float(LOYO["auc_m1"].mean()),
        "loyo_auc_m2":  float(LOYO["auc_m2"].mean()),
        "loyo_auc_m3":  float(LOYO["auc_m3"].mean()),
        "test2025_auc": auc_te, "test2025_ap": ap_te,
    })
    mlflow.log_dict(res, "upset/config_and_results.json")
    for p in OUT.glob("*.json"):
        mlflow.log_artifact(str(p), "upset")

print("完了")

test 2025: AUC=0.6357  AP=0.1603  n=47,497  pos_rate=0.1067


,上位k頭,単勝回収率,複勝回収率
0,1,0.7803,0.6550
1,2,0.7136,0.6677
2,3,0.7575,0.6858


🏃 View run tp-24-upset-v5-no-oof at: http://127.0.0.1:23230/#/experiments/26/runs/3fa15195831940a7b98a66e698755969
🧪 View experiment at: http://127.0.0.1:23230/#/experiments/26
完了
